# Transductive vs. Inductive Evaluation Gap

This notebook reproduces the **Transductive vs. Inductive Evaluation Gap** experiment from the `gnn-fraud` repository.

Both runs use the **same** GraphSAGE model, seed, hyperparameters, and loss. They differ only in the adjacency used for message passing during training:

- **Transductive** — train on the full Elliptic graph (edges touching test-period nodes are visible).
- **Inductive**    — train only on edges whose both endpoints are in time steps 1–34; the test-period adjacency is restored at inference.

The F1 gap between the two quantifies how much of the published GraphSAGE performance comes from **structural leakage** rather than from genuine inductive generalisation.

## 1. Install dependencies

Colab already ships with PyTorch. We add `torch-geometric` and the classical ML stack the repo depends on.

In [ ]:
!pip install -q torch-geometric==2.5.3
!pip install -q scikit-learn pandas numpy

## 2. Clone the repository

In [ ]:
import os
if not os.path.exists('gnn-fraud'):
    !git clone https://github.com/Saket-Maganti/gnn-fraud.git
%cd gnn-fraud

## 3. Download the Elliptic Bitcoin Dataset

The dataset is available on Kaggle as `ellipticco/elliptic-data-set`. Upload your `kaggle.json` API token when prompted.

In [ ]:
import os, pathlib

raw_dir = pathlib.Path('data/raw')
raw_dir.mkdir(parents=True, exist_ok=True)

expected = ['elliptic_txs_features.csv', 'elliptic_txs_classes.csv', 'elliptic_txs_edgelist.csv']
missing  = [f for f in expected if not (raw_dir / f).exists()]

if missing:
    print('Missing CSVs, pulling from Kaggle...')
    !pip install -q kaggle
    from google.colab import files  # type: ignore
    if not pathlib.Path('/root/.kaggle/kaggle.json').exists():
        print('Upload your kaggle.json API token:')
        files.upload()
        !mkdir -p /root/.kaggle && mv kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json
    !kaggle datasets download ellipticco/elliptic-data-set -p data/raw --unzip
else:
    print('Dataset already present.')

## 4. Run the transductive experiment

In [ ]:
!python experiments/run_transductive.py --epochs 200 --seed 42 --device auto

## 5. Run the inductive experiment

In [ ]:
!python experiments/run_inductive.py --epochs 200 --seed 42 --device auto

## 6. Compare results

In [ ]:
!python experiments/compare_leakage_gap.py

## 7. Inspect per-timestep breakdown (optional)

Both runners also save a per-timestep F1 breakdown for time steps 35–49 inside the result JSONs.

In [ ]:
import json

for name in ['transductive', 'inductive']:
    path = f'results/{name}_results.json'
    with open(path) as f:
        r = json.load(f)
    print(f"\n{name.upper():<13s}  best F1 = {r['best_metrics']['f1']:.4f}")
    print(f"{'step':>4s} {'f1':>8s} {'prec':>8s} {'recall':>8s}")
    for step, m in sorted(r['per_timestep'].items(), key=lambda kv: int(kv[0])):
        print(f"{step:>4s} {m['f1']:>8.4f} {m['precision']:>8.4f} {m['recall']:>8.4f}")